In [1]:
import pulp


In [3]:
model = pulp.LpProblem("NutribBox_Optimisation", pulp.LpMinimize)

## Datasets
factories = ["F1","F2","F3","F4"]
dcs = ["DC1","DC2","DC3"]
destinations = ["D1","D2","D3","D4","D5"]

##Supply and demand constraints
supply = {"F1":180, "F2":140, "F3":160, "F4":120}
demand = {"D1":110, "D2":130, "D3":140,  "D4":90,  "D5":130 }

In [5]:
##adding the route data and constraints
#Data in the tuple (Cost, Time, CO2, Capacity)

#Factory to Distribution Centre
factory_dc = {
    ("F1","DC1"):(14, 1.2, 6.2, 120), ("F1","DC2"):(12, 1.5, 7.1, 120), ("F1","DC3"):(16, 1.0, 6.5, 120),
    ("F2","DC1"):(13, 1.1, 6.0, 120), ("F2","DC2"):(11, 1.6, 7.4, 120), ("F2","DC3"):(15, 1.2, 6.6, 120),
    ("F3","DC1"):(10, 1.4, 7.6, 120), ("F3","DC2"):(9, 1.7, 8.2, 120), ("F3","DC3"):(12, 1.3, 7.4, 120),
    ("F4","DC1"):(8, 1.6, 8.0, 120), ("F4","DC2"):(7, 1.8, 8.5, 120), ("F4","DC3"):(9, 1.5, 8.1, 120)  
    
}

#Distribution Center to Destination
dc_destination = {
    ("DC1","D1"):(9, 0.9, 5.8, 120), ("DC1","D2"):(11, 1.0, 6.1, 120), ("DC1","D3"):(12, 1.2, 6.4, 120), ("DC1","D4"):(10, 1.1, 6.0, 120), ("DC1","D5"):(13, 1.3, 6.6, 120),
    ("DC2","D1"):(7, 0.8, 6.7, 120), ("DC2","D2"):(8, 0.9, 6.9, 120), ("DC2","D3"):(10, 1.0, 7.2, 120), ("DC2","D4"):(9, 1.0, 7.0, 120), ("DC2","D5"):(11, 1.1, 7.4, 120),
    ("DC3","D1"):(12, 0.7, 6.2, 120), ("DC3","D2"):(10, 0.8, 6.0, 120), ("DC3","D3"):(9, 0.9, 6.3, 120), ("DC3","D4"):(8, 1.0, 6.5, 120), ("DC3","D5"):(10, 1.1, 6.8, 120)
    
}

# Factories to Destination(Direct route by train)

factory_dest = {
    ("F1","D1"):(28, 2.2, 4.0, 300), ("F1","D2"):(30, 2.4, 4.2, 300), 
    ("F2","D1"):(27, 2.3, 4.1, 300), ("F2","D2"):(29, 2.5, 4.3, 300) 
}



In [7]:
#Ensuring that the pallets are moved as 1 and not split
# X ensure factory to destination, Y ensures DCs to destinations, Z ensures factories to destination

X = pulp.LpVariable.dicts("Factory to Distribution centres", factory_dc.keys(), lowBound= 0, cat= "Integer")
Y = pulp.LpVariable.dicts("Distribution centres to Destination", dc_destination.keys(), lowBound=0, cat= "Integer")
Z = pulp.LpVariable.dicts("Factory to Destination", factory_dest.keys(), lowBound= 0, cat= "Integer")

#Binary activation for the 8 active lanes for factory to DCs to reduce complexity
binary =  pulp.LpVariable.dicts("Active Lane", factory_dc.keys(), cat= "Binary")


# Objective function
model += (
    pulp.lpSum([X[r] * factory_dc[r][0] for r in factory_dc]) + pulp.lpSum([Y[r] * dc_destination[r][0] for r in dc_destination]) + 
    pulp.lpSum([Z[r] * factory_dest[r][0] for r in factory_dest])
)

In [14]:
# Constraints

# Supply Constraint
for f in factories:
    f_dc_sum = pulp.lpSum([X[r] for r in factory_dc if r[0] == f])
    f_dest_sum = pulp.lpSum([Z[r] for r in factory_dest if r[0] == f])
    model += (f_dc_sum + f_dest_sum == supply[f])

# Demand Constraint

for d in destinations:
    dc_dest_sum = pulp.lpSum([Y[r] for r in dc_destination if r[1] == d])
    f_d_sum = pulp.lpSum([Z[r] for r in factory_dest if r[1] == d])
    model += (dc_dest_sum + f_d_sum == demand[d])

# Transshipment Balance(DC In =DC Out)

for dc in dcs:
    model += (pulp.lpSum([X[r] for r in factory_dc if r[1] == dc]) == pulp.lpSum([Y[r] for r in dc_destination if r[0] == dc]) )

# Arc Capacities

for r in factory_dc: model += (X[r] <= factory_dc[r][3])
for r in dc_destination: model += (Y[r] <= dc_destination[r][3])
for r in factory_dest: model += (Z[r] <= factory_dest[r][3])

#Constraint for DC2 
model += (pulp.lpSum([X[r] for r in factory_dc if r[1] == "DC2"]) <= 150)

# Average time Constraint
# Time constraaint =  flow(total pallets to be distributed) * average time

time_cost = (
    pulp.lpSum([X[r] * factory_dc[r][1] for r in factory_dc]) + pulp.lpSum([Y[r] * dc_destination[r][1] for r in dc_destination]) +
    pulp.lpSum([Z[r] * factory_dest[r][1] for r in factory_dest]) )                                                              
model += (time_cost <= 2.1 * 600)

# Carbon footprint constraint
# Total emission = 3900

emissions = (
    pulp.lpSum([X[r] * factory_dc[r][2] for r in factory_dc]) + pulp.lpSum([Y[r] * dc_destination[r][2] for r in dc_destination]) +
    pulp.lpSum([Z[r] * factory_dest[r][2] for r in factory_dest]) )
model += (emissions <= 3900)

# Risk diversification (no DC should have > 270 pallets)
for dc in dcs:
    model += (pulp.lpSum([Y[r] for r in dc_destination if r[0] == dc]) <= 270)

# Poliy for minimum rail capaity (direct route to destination should be >= 120)
model += (pulp.lpSum([Z[r] for r in factory_dest]) >= 120 )

# Operational coplexity reduction ( maximum of 8 active factories to Dc lanes)
# We will have to link the integer flow of X variable(factory to DC total) to the binary constraint binary. if X > 0, B must be 1.
for r in factory_dc:
          model += (X[r] <= factory_dc[r][3] * binary[r])
model += (pulp.lpSum([binary[r] for r in binary]) <= 8)

# time to solve
model.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"total Minimised Cost: ${pulp.value(model.objective):,}")




total Minimised Cost: $16,903.333300000002


In [18]:
# 8. Detailed Output Breakdown
if pulp.LpStatus[model.status] == "Optimal":
    print("-" * 75)
    print(f"{"ROUTE TYPE":<20} | {"PATH":<12} | {"PALLETS":<8} | {"COST":<8} | {"TIME":<6} | {"CO2":<6}")
    print("-" * 75)
    
    # Track metrics for validation
    total_pallets = 0
    total_time = 0
    total_co2 = 0

    # Factory to DC
    for r in factory_dc:
        val = pulp.value(X[r])
        if val > 0:
            c, t, e, _ = factory_dc[r]
            print(f"{"Factory -> DC":<20} | {str(r):<12} | {int(val):<8} | ${c*val:<7} | {t:<6} | {e:<6}")
            total_pallets += val
            total_time += val * t
            total_co2 += val * e

    # DC to Destination
    for r in dc_destination:
        val = pulp.value(Y[r])
        if val > 0:
            c, t, e, _ = dc_destination[r]
            print(f"{"DC -> Destination":<20} | {str(r):<12} | {int(val):<8} | ${c*val:<7} | {t:<6} | {e:<6}")
            # Note: We don't add to total_pallets here to avoid double counting (it's the same 600 pallets)
            total_time += val * t
            total_co2 += val * e

    # Direct destination route
    for r in factory_dest:
        val = pulp.value(Z[r])
        if val > 0:
            c, t, e, _ = factory_dest[r]
            print(f"{"Direct destination(rail)":<20} | {str(r):<12} | {int(val):<8} | ${c*val:<7} | {t:<6} | {e:<6}")
            total_pallets += val
            total_time += val * t
            total_co2 += val * e

    print("-" * 75)
    print(f"FINAL RESULTS:")
    print(f"Total Minimized Cost: ${pulp.value(model.objective):,.0f}")
    print(f"Average Lead Time:    {total_time/600:.2f} Days (Limit: 2.1)")
    print(f"Total Emissions:      {total_co2:.1f} kg CO2 (Limit: 3900)")
    print(f"Active F->DC Lanes:   {sum(pulp.value(B[r]) for r in B):.0f} (Limit: 8)")
else:
    print("No optimal solution found. Check constraints.")

    ### Adjust this to show you the infeasibility.

No optimal solution found. Check constraints.
